# Hotel Reviews - BERTopic Modeling

This notebook implements BERTopic, a modern topic modeling approach using transformers.

**Goals:**
1. Load preprocessed review data (using original text, not lemmatized)
2. Generate semantic embeddings using sentence transformers
3. Train BERTopic models for negative and positive reviews
4. Extract and interpret topics
5. Compare with LDA results
6. Visualize topics with BERTopic's built-in visualizations
7. Save models and results

**Why BERTopic?**
- Better semantic understanding than LDA
- Auto-determines optimal number of topics
- Handles short texts better
- More interpretable topics

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Imports

import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# BERTopic and related libraries
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
plt.style.use('seaborn-v0_8-whitegrid')

%matplotlib inline

# Set random seed
np.random.seed(42)

print("Imports successful!")

KeyboardInterrupt: 

## 1. Load Preprocessed Data

**Important:** BERTopic works better with original text (not lemmatized) as transformers understand context.

In [ ]:
# Option 1: Load sample data (recommended for first run)
USE_SAMPLE = True  # Set to False to use full dataset

if USE_SAMPLE:
    print("Loading SAMPLE datasets (50K reviews each)...")
    df_neg = pd.read_parquet('data/negative_reviews_sample.parquet')
    df_pos = pd.read_parquet('data/positive_reviews_sample.parquet')
else:
    print("Loading FULL datasets...")
    print("⚠️  Warning: This may take significant time and memory!")
    df_neg = pd.read_parquet('data/negative_reviews_processed.parquet')
    df_pos = pd.read_parquet('data/positive_reviews_processed.parquet')

print(f"\nNegative reviews: {len(df_neg):,}")
print(f"Positive reviews: {len(df_pos):,}")

# Use original text (not lemmatized)
documents_neg = df_neg['text_original'].tolist()
documents_pos = df_pos['text_original'].tolist()

print(f"\nSample negative review:")
print(f"  {documents_neg[0][:200]}...")
print(f"\nSample positive review:")
print(f"  {documents_pos[0][:200]}...")

## 2. Configure BERTopic Components

In [ ]:
# 1. Embedding Model
print("Loading embedding model...")
print("  Using: all-MiniLM-L6-v2 (fast, good quality)")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Dimensionality Reduction (UMAP)
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

# 3. Clustering (HDBSCAN)
hdbscan_model = HDBSCAN(
    min_cluster_size=50,      # Minimum reviews per topic
    min_samples=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

# 4. Vectorizer (for topic representation)
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    stop_words='english'
)

print("\n✅ Components configured!")

## 3. Train BERTopic Model - Negative Reviews

In [ ]:
print("Training BERTopic model for NEGATIVE reviews...")
print("This may take 5-15 minutes depending on dataset size.\n")

# Initialize BERTopic
topic_model_neg = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=15,
    verbose=True,
    calculate_probabilities=True
)

# Fit model
topics_neg, probs_neg = topic_model_neg.fit_transform(documents_neg)

print(f"\n✅ Model trained!")
print(f"  Number of topics discovered: {len(set(topics_neg)) - 1}")  # -1 for outliers
print(f"  Outliers (topic -1): {sum(1 for t in topics_neg if t == -1)}")

In [ ]:
# Get topic information
topic_info_neg = topic_model_neg.get_topic_info()

print("\nNegative Review Topics:")
print(topic_info_neg[topic_info_neg['Topic'] != -1][['Topic', 'Count', 'Name']].head(15))

## 4. Train BERTopic Model - Positive Reviews

In [ ]:
print("Training BERTopic model for POSITIVE reviews...")
print("This may take 5-15 minutes depending on dataset size.\n")

# Initialize BERTopic
topic_model_pos = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=15,
    verbose=True,
    calculate_probabilities=True
)

# Fit model
topics_pos, probs_pos = topic_model_pos.fit_transform(documents_pos)

print(f"\n✅ Model trained!")
print(f"  Number of topics discovered: {len(set(topics_pos)) - 1}")  # -1 for outliers
print(f"  Outliers (topic -1): {sum(1 for t in topics_pos if t == -1)}")

In [ ]:
# Get topic information
topic_info_pos = topic_model_pos.get_topic_info()

print("\nPositive Review Topics:")
print(topic_info_pos[topic_info_pos['Topic'] != -1][['Topic', 'Count', 'Name']].head(15))

## 5. Display Topics in Detail

In [ ]:
def display_bertopic_topics(topic_model, sentiment='negative', num_topics=15):
    """
    Display BERTopic topics with their keywords.
    
    Args:
        topic_model: Trained BERTopic model
        sentiment: 'negative' or 'positive'
        num_topics: Number of topics to display
    """
    print("="*100)
    print(f"{sentiment.upper()} REVIEW TOPICS (BERTopic)")
    print("="*100)
    
    topics = topic_model.get_topics()
    topic_ids = sorted([t for t in topics.keys() if t != -1])[:num_topics]
    
    for topic_id in topic_ids:
        words = topic_model.get_topic(topic_id)
        if words:
            top_words = [word for word, score in words[:10]]
            print(f"\nTopic {topic_id}:")
            print(f"  Keywords: {', '.join(top_words)}")
            
            # Show representative documents
            repr_docs = topic_model.get_representative_docs(topic_id)
            if repr_docs:
                print(f"  Example: {repr_docs[0][:150]}...")

display_bertopic_topics(topic_model_neg, sentiment='negative')
print("\n")
display_bertopic_topics(topic_model_pos, sentiment='positive')

## 6. Assign Topic Labels (Manual Interpretation)

Based on the keywords, assign meaningful labels to topics.

In [ ]:
# TODO: Update these labels based on the actual topics discovered
# This is a template - you'll need to customize based on your results

topic_labels_neg_bert = {
    0: "Room Size Issues",
    1: "Noise Problems",
    2: "Breakfast Complaints",
    3: "Bathroom Issues",
    4: "Staff Service Problems",
    5: "Cleanliness Concerns",
    6: "Location/Transport Issues",
    7: "Price/Value Concerns",
    8: "Facilities/Amenities",
    9: "Check-in/Reception",
    # Add more as needed
}

topic_labels_pos_bert = {
    0: "Excellent Location",
    1: "Friendly Staff",
    2: "Comfortable Rooms",
    3: "Great Breakfast",
    4: "Clean & Well-maintained",
    5: "Modern Facilities",
    6: "Value for Money",
    7: "Beautiful Views",
    8: "Convenient Location",
    9: "Helpful Staff",
    # Add more as needed
}

# Update topic model with custom labels
topic_model_neg.set_topic_labels(topic_labels_neg_bert)
topic_model_pos.set_topic_labels(topic_labels_pos_bert)

print("✅ Topic labels assigned!")

## 7. BERTopic Visualizations

In [ ]:
# 1. Visualize topics (2D projection)
print("Creating topic visualization (2D)...")
fig_neg = topic_model_neg.visualize_topics()
fig_neg.write_html('outputs/visualizations/intertopic_distance/bertopic_negative_interactive.html')
fig_neg.show()

print("\nSaved to: outputs/visualizations/intertopic_distance/bertopic_negative_interactive.html")

In [ ]:
# Topic visualization for positive reviews
print("Creating topic visualization (2D)...")
fig_pos = topic_model_pos.visualize_topics()
fig_pos.write_html('outputs/visualizations/intertopic_distance/bertopic_positive_interactive.html')
fig_pos.show()

print("\nSaved to: outputs/visualizations/intertopic_distance/bertopic_positive_interactive.html")

In [ ]:
# 2. Visualize topic hierarchy
print("Creating hierarchical topic view...")
try:
    hierarchical_topics_neg = topic_model_neg.hierarchical_topics(documents_neg)
    fig_hier_neg = topic_model_neg.visualize_hierarchy(hierarchical_topics=hierarchical_topics_neg)
    fig_hier_neg.write_html('outputs/visualizations/topic_distributions/bertopic_negative_hierarchy.html')
    fig_hier_neg.show()
except ValueError as e:
    print(f"Skipping hierarchy visualization (known BERTopic bug): {e}")
    print("This does not affect the topic model results.")

In [ ]:
# Hierarchy for positive reviews
print("Creating hierarchical topic view...")
try:
    hierarchical_topics_pos = topic_model_pos.hierarchical_topics(documents_pos)
    fig_hier_pos = topic_model_pos.visualize_hierarchy(hierarchical_topics=hierarchical_topics_pos)
    fig_hier_pos.write_html('outputs/visualizations/topic_distributions/bertopic_positive_hierarchy.html')
    fig_hier_pos.show()
except ValueError as e:
    print(f"Skipping hierarchy visualization (known BERTopic bug): {e}")
    print("This does not affect the topic model results.")

In [ ]:
# 3. Visualize topic word scores (bar chart)
print("Creating topic bar charts...")
fig_barchart_neg = topic_model_neg.visualize_barchart(top_n_topics=10, n_words=10)
fig_barchart_neg.write_html('outputs/visualizations/topic_distributions/bertopic_negative_barchart.html')
fig_barchart_neg.show()

In [ ]:
# Bar chart for positive reviews
print("Creating topic bar charts...")
fig_barchart_pos = topic_model_pos.visualize_barchart(top_n_topics=10, n_words=10)
fig_barchart_pos.write_html('outputs/visualizations/topic_distributions/bertopic_positive_barchart.html')
fig_barchart_pos.show()

## 8. Create Word Clouds for BERTopic Topics

In [ ]:
def create_bertopic_wordclouds(topic_model, sentiment='negative', num_topics=10):
    """
    Create word clouds for BERTopic topics.
    
    Args:
        topic_model: Trained BERTopic model
        sentiment: 'negative' or 'positive'
        num_topics: Number of topics to visualize
    """
    colormap = 'Reds' if sentiment == 'negative' else 'Greens'
    
    topics = topic_model.get_topics()
    topic_ids = sorted([t for t in topics.keys() if t != -1])[:num_topics]
    
    # Calculate grid
    cols = 3
    rows = int(np.ceil(len(topic_ids) / cols))
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = axes.flatten() if len(topic_ids) > 1 else [axes]
    
    for idx, topic_id in enumerate(topic_ids):
        # Get topic words and scores
        words = topic_model.get_topic(topic_id)
        if not words:
            continue
        
        # Create frequency dict for word cloud
        word_freq = {word: score for word, score in words[:50]}
        
        # Create word cloud
        wc = WordCloud(
            width=800,
            height=400,
            background_color='white',
            colormap=colormap,
            relative_scaling=0.5,
            min_font_size=10
        ).generate_from_frequencies(word_freq)
        
        # Plot
        axes[idx].imshow(wc, interpolation='bilinear')
        axes[idx].axis('off')
        
        # Get topic label
        topic_info = topic_model.get_topic_info()
        topic_row = topic_info[topic_info['Topic'] == topic_id]
        if not topic_row.empty:
            label = topic_row.iloc[0]['Name']
            axes[idx].set_title(f'Topic {topic_id}: {label}', 
                              fontsize=11, fontweight='bold')
        
        # Save individual
        individual_fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'{sentiment.capitalize()} Reviews - Topic {topic_id}',
                    fontsize=14, fontweight='bold')
        individual_fig.tight_layout()
        individual_fig.savefig(
            f'outputs/visualizations/wordclouds/bertopic_{sentiment}_topic{topic_id}.png',
            dpi=150,
            bbox_inches='tight'
        )
        plt.close(individual_fig)
    
    # Hide unused subplots
    for idx in range(len(topic_ids), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(
        f'outputs/visualizations/wordclouds/bertopic_{sentiment}_all_topics.png',
        dpi=150,
        bbox_inches='tight'
    )
    plt.show()

print("Creating word clouds for negative reviews...")
create_bertopic_wordclouds(topic_model_neg, sentiment='negative', num_topics=10)

print("\nCreating word clouds for positive reviews...")
create_bertopic_wordclouds(topic_model_pos, sentiment='positive', num_topics=10)

## 9. Topic Distribution Analysis

In [ ]:
# Add topics to dataframes
df_neg['bertopic_topic'] = topics_neg
df_pos['bertopic_topic'] = topics_pos

# Count distributions (excluding outliers)
topic_dist_neg = df_neg[df_neg['bertopic_topic'] != -1]['bertopic_topic'].value_counts().sort_index()
topic_dist_pos = df_pos[df_pos['bertopic_topic'] != -1]['bertopic_topic'].value_counts().sort_index()

print("Negative Reviews - Topic Distribution:")
print(topic_dist_neg.head(10))
print(f"\nOutliers: {sum(df_neg['bertopic_topic'] == -1)}")

print("\nPositive Reviews - Topic Distribution:")
print(topic_dist_pos.head(10))
print(f"\nOutliers: {sum(df_pos['bertopic_topic'] == -1)}")

In [ ]:
# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Negative reviews (top 10 topics)
ax1 = axes[0]
top_neg = topic_dist_neg.head(10)
topic_names_neg = [topic_info_neg[topic_info_neg['Topic'] == t]['Name'].values[0] 
                   if t in topic_info_neg['Topic'].values else f'Topic {t}' 
                   for t in top_neg.index]
ax1.barh(topic_names_neg, top_neg.values, color='salmon', edgecolor='white')
ax1.set_xlabel('Number of Reviews', fontsize=12)
ax1.set_title('Negative Reviews - Top 10 Topics (BERTopic)', fontsize=13, fontweight='bold')
ax1.invert_yaxis()

# Positive reviews (top 10 topics)
ax2 = axes[1]
top_pos = topic_dist_pos.head(10)
topic_names_pos = [topic_info_pos[topic_info_pos['Topic'] == t]['Name'].values[0]
                   if t in topic_info_pos['Topic'].values else f'Topic {t}'
                   for t in top_pos.index]
ax2.barh(topic_names_pos, top_pos.values, color='lightgreen', edgecolor='white')
ax2.set_xlabel('Number of Reviews', fontsize=12)
ax2.set_title('Positive Reviews - Top 10 Topics (BERTopic)', fontsize=13, fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig('outputs/visualizations/topic_distributions/bertopic_topic_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 10. Save Models and Results

In [ ]:
# Save BERTopic models
print("Saving BERTopic models...")
topic_model_neg.save('outputs/models/bertopic_negative', serialization="safetensors")
topic_model_pos.save('outputs/models/bertopic_positive', serialization="safetensors")

# Save topic labels
with open('outputs/models/topic_labels_negative_bert.pkl', 'wb') as f:
    pickle.dump(topic_labels_neg_bert, f)

with open('outputs/models/topic_labels_positive_bert.pkl', 'wb') as f:
    pickle.dump(topic_labels_pos_bert, f)

print("\n✅ Models saved successfully!")

In [ ]:
# Export topic information to CSV
print("Exporting topic information...")

# Negative topics
topic_info_neg_export = topic_info_neg[topic_info_neg['Topic'] != -1].copy()
topic_info_neg_export.to_csv('outputs/topics/bertopic_negative_topics.csv', index=False)

# Positive topics
topic_info_pos_export = topic_info_pos[topic_info_pos['Topic'] != -1].copy()
topic_info_pos_export.to_csv('outputs/topics/bertopic_positive_topics.csv', index=False)

print("\n✅ Topic information exported!")
print("  - outputs/topics/bertopic_negative_topics.csv")
print("  - outputs/topics/bertopic_positive_topics.csv")

## 11. Compare with LDA Results

In [ ]:
# Load LDA topics if available
try:
    lda_topics_neg = pd.read_csv('outputs/topics/lda_negative_topics.csv')
    lda_topics_pos = pd.read_csv('outputs/topics/lda_positive_topics.csv')
    
    print("LDA vs BERTopic Comparison")
    print("="*80)
    print(f"\nNegative Reviews:")
    print(f"  LDA Topics: {len(lda_topics_neg)}")
    print(f"  BERTopic Topics: {len(topic_info_neg_export)}")
    print(f"  BERTopic Outliers: {sum(df_neg['bertopic_topic'] == -1)}")
    
    print(f"\nPositive Reviews:")
    print(f"  LDA Topics: {len(lda_topics_pos)}")
    print(f"  BERTopic Topics: {len(topic_info_pos_export)}")
    print(f"  BERTopic Outliers: {sum(df_pos['bertopic_topic'] == -1)}")
    
    print("\n" + "="*80)
    print("\nKey Observations:")
    print("  - BERTopic auto-determines number of topics (no manual tuning needed)")
    print("  - BERTopic topics tend to be more semantically coherent")
    print("  - LDA provides more evenly distributed topics")
    print("  - BERTopic identifies outliers that don't fit any topic")
    
except FileNotFoundError:
    print("LDA results not found. Run 03_topic_modeling_lda.ipynb first for comparison.")

## 12. Summary and Next Steps

In [ ]:
print("="*80)
print("BERTOPIC MODELING SUMMARY")
print("="*80)
print(f"\nNegative Reviews:")
print(f"  - Documents analyzed: {len(documents_neg):,}")
print(f"  - Topics discovered: {len(set(topics_neg)) - 1}")
print(f"  - Outliers: {sum(1 for t in topics_neg if t == -1):,}")
print(f"  - Embedding model: all-MiniLM-L6-v2")

print(f"\nPositive Reviews:")
print(f"  - Documents analyzed: {len(documents_pos):,}")
print(f"  - Topics discovered: {len(set(topics_pos)) - 1}")
print(f"  - Outliers: {sum(1 for t in topics_pos if t == -1):,}")
print(f"  - Embedding model: all-MiniLM-L6-v2")

print(f"\nOutput Files:")
print(f"  - Models: outputs/models/bertopic_negative/, bertopic_positive/")
print(f"  - Topics: outputs/topics/bertopic_*_topics.csv")
print(f"  - Visualizations: outputs/visualizations/")
print(f"    * Word clouds: Multiple PNG files")
print(f"    * Interactive HTML: 6+ files")

print("\n" + "="*80)
print("✅ BERTopic Modeling Complete!")
print("="*80)
print("\nNext steps:")
print("  1. Compare BERTopic vs LDA results")
print("  2. Refine topic labels based on representative documents")
print("  3. Generate business insights and recommendations")
print("  4. Create final report with key findings")